# Exercise 08 — ComplexNumber-klassen

NUMA01 VT2026 — Arvid Brenner & Sixten Midsem

Lösningar till `exercise08.md`. Vi bygger upp en `ComplexNumber`-klass i
flera steg och jämför mot Pythons inbyggda `complex` på slutet.


In [1]:
import numpy as np


## Task 2 — `RationalNumber.__sub__`

Boken (Chap. 8) presenterar klassen `RationalNumber`. Subtraktion
$\frac{a}{b} - \frac{c}{d} = \frac{ad - bc}{bd}$.


In [2]:
class RationalNumber:
    def __init__(self, numerator, denominator):
        self.numerator = numerator
        self.denominator = denominator

    def add(self, other):
        p1, q1 = self.numerator, self.denominator
        p2, q2 = other.numerator, other.denominator
        return RationalNumber(p1*q2 + p2*q1, q1*q2)

    def __sub__(self, other):
        p1, q1 = self.numerator, self.denominator
        p2, q2 = other.numerator, other.denominator
        return RationalNumber(p1*q2 - p2*q1, q1*q2)

    def __repr__(self):
        return f"{self.numerator}/{self.denominator}"


a = RationalNumber(2, 3)
b = RationalNumber(1, 4)
print(a - b, "  (förväntat 5/12)")


5/12   (förväntat 5/12)


## Task 3–10 — `ComplexNumber`

Vi samlar alla metoder i en klass.


In [3]:
import math


class ComplexNumber:
    """Egen implementation av komplexa tal: a + i*b."""

    def __init__(self, real, imag=0.0):
        self._real = float(real)
        self._imag = float(imag)

    # Task 4
    def real(self):
        return self._real

    def imag(self):
        return self._imag

    # Task 5
    def is_real(self):
        return self._imag == 0.0

    def is_imaginary(self):
        return self._real == 0.0 and self._imag != 0.0

    # Task 6
    def __repr__(self):
        a, b = self._real, self._imag
        if b == 0:
            return f"{a}"
        sign = '+' if b >= 0 else '-'
        return f"{a} {sign} i*{abs(b)}"

    # Task 7
    def abs(self):
        return math.hypot(self._real, self._imag)

    def arg(self):
        return math.atan2(self._imag, self._real)

    # Task 8
    def __eq__(self, other):
        other = self._coerce(other)
        return math.isclose(self._real, other._real) and \
               math.isclose(self._imag, other._imag, abs_tol=1e-12)

    def __hash__(self):
        return hash((self._real, self._imag))

    # ------------------------------------------------------------------
    # Hjälp: gör så att vi kan blanda ComplexNumber med int/float
    # ------------------------------------------------------------------
    @staticmethod
    def _coerce(other):
        if isinstance(other, ComplexNumber):
            return other
        if isinstance(other, (int, float)):
            return ComplexNumber(other, 0)
        raise TypeError(f"Kan inte tolka {type(other).__name__} som ComplexNumber")

    # Task 9 — aritmetik
    def __add__(self, other):
        o = self._coerce(other)
        return ComplexNumber(self._real + o._real, self._imag + o._imag)
    __radd__ = __add__

    def __sub__(self, other):
        o = self._coerce(other)
        return ComplexNumber(self._real - o._real, self._imag - o._imag)

    def __rsub__(self, other):
        return self._coerce(other) - self

    def __mul__(self, other):
        o = self._coerce(other)
        a, b = self._real, self._imag
        c, d = o._real, o._imag
        return ComplexNumber(a*c - b*d, a*d + b*c)
    __rmul__ = __mul__

    def __truediv__(self, other):
        o = self._coerce(other)
        a, b = self._real, self._imag
        c, d = o._real, o._imag
        denom = c*c + d*d
        return ComplexNumber((a*c + b*d) / denom, (b*c - a*d) / denom)

    def __rtruediv__(self, other):
        return self._coerce(other) / self

    def __pow__(self, n):
        """Heltalspotens via upprepad multiplikation. Polar form för float."""
        if isinstance(n, int):
            if n == 0:
                return ComplexNumber(1, 0)
            if n < 0:
                return ComplexNumber(1, 0) / (self ** (-n))
            result = ComplexNumber(1, 0)
            for _ in range(n):
                result = result * self
            return result
        # generell potens via polar form
        r = self.abs()
        phi = self.arg()
        return ComplexNumber(
            (r ** n) * math.cos(n * phi),
            (r ** n) * math.sin(n * phi),
        )

    def __neg__(self):
        return ComplexNumber(-self._real, -self._imag)


## Task 10 — Jämför mot Pythons `complex`


In [4]:
def to_py(z):
    """ComplexNumber → Python complex för jämförelse."""
    return complex(z.real(), z.imag())


z1 = ComplexNumber(2, 3)
z2 = ComplexNumber(1, -4)

p1 = complex(2, 3)
p2 = complex(1, -4)

# repr
print("z1 =", z1, "  (Python:", p1, ")")

# delar
print("real:", z1.real(), p1.real)
print("imag:", z1.imag(), p1.imag)
print("|z1|:", z1.abs(), abs(p1))
print("arg :", z1.arg(), math.atan2(p1.imag, p1.real))

# is_real / is_imaginary
print("ComplexNumber(5).is_real()        :", ComplexNumber(5).is_real())
print("ComplexNumber(0, 7).is_imaginary():", ComplexNumber(0, 7).is_imaginary())

# aritmetik
for op_name, op in [('add', lambda a, b: a + b),
                    ('sub', lambda a, b: a - b),
                    ('mul', lambda a, b: a * b),
                    ('div', lambda a, b: a / b)]:
    mine = op(z1, z2)
    ref = op(p1, p2)
    print(f"{op_name}: mine={to_py(mine)}  python={ref}  match={to_py(mine) == ref}")

# blandad: ComplexNumber + int, float * ComplexNumber, etc.
print("\nBlandade operationer:")
print("z1 + 5 =", z1 + 5,    "  (python:", p1 + 5, ")")
print("5 + z1 =", 5 + z1,    "  (python:", 5 + p1, ")")
print("2.5 * z1 =", 2.5 * z1, "  (python:", 2.5 * p1, ")")
print("z1 / 2 =", z1 / 2,    "  (python:", p1 / 2, ")")
print("10 / z1 =", 10 / z1,  "  (python:", 10 / p1, ")")

# Potens (heltal)
print("\nz1 ** 3 =", z1 ** 3, "  (python:", p1 ** 3, ")")
print("z1 ** -2 =", z1 ** -2, "  (python:", p1 ** -2, ")")

# Likhet
print("\nComplexNumber(2, 3) == ComplexNumber(2, 3):",
      ComplexNumber(2, 3) == ComplexNumber(2, 3))
print("ComplexNumber(5, 0) == 5                  :",
      ComplexNumber(5, 0) == 5)


z1 = 2.0 + i*3.0   (Python: (2+3j) )
real: 2.0 2.0
imag: 3.0 3.0
|z1|: 3.605551275463989 3.605551275463989
arg : 0.982793723247329 0.982793723247329
ComplexNumber(5).is_real()        : True
ComplexNumber(0, 7).is_imaginary(): True
add: mine=(3-1j)  python=(3-1j)  match=True
sub: mine=(1+7j)  python=(1+7j)  match=True
mul: mine=(14-5j)  python=(14-5j)  match=True
div: mine=(-0.5882352941176471+0.6470588235294118j)  python=(-0.5882352941176471+0.6470588235294118j)  match=True

Blandade operationer:
z1 + 5 = 7.0 + i*3.0   (python: (7+3j) )
5 + z1 = 7.0 + i*3.0   (python: (7+3j) )
2.5 * z1 = 5.0 + i*7.5   (python: (5+7.5j) )
z1 / 2 = 1.0 + i*1.5   (python: (1+1.5j) )
10 / z1 = 1.5384615384615385 - i*2.3076923076923075   (python: (1.5384615384615383-2.307692307692308j) )

z1 ** 3 = -46.0 + i*9.0   (python: (-46+9j) )
z1 ** -2 = -0.029585798816568046 - i*0.07100591715976332   (python: (-0.029585798816568046-0.07100591715976332j) )

ComplexNumber(2, 3) == ComplexNumber(2, 3): True
ComplexNumb